# Video Emotion Recognition Test
This notebook loads your trained model from Google Drive (or local path) and runs inference on a test video.

In [1]:
# Install requirements if running in Colab
!pip install tensorflow opencv-python-headless numpy

In [2]:
# Mount Google Drive if you are running this in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted.")
except ImportError:
    print("Not running in Colab, skipping Drive mount.")

Mounted at /content/drive
Google Drive mounted.


In [3]:
from google.colab import files
uploaded = files.upload()

In [5]:
import cv2
import numpy as np
import tensorflow as tf
from IPython.display import HTML, display
from base64 import b64encode
from pathlib import Path
import os
import shutil
import subprocess

# ---- Manual overrides -------------------------------------------------------
# Leave as None for auto-detection, or set an explicit path string.
MODEL_PATH = None
INPUT_VIDEO = None
OUTPUT_VIDEO = "output_emotion.mp4"
PRESERVE_AUDIO = True

emotion_labels = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd

MODEL_CANDIDATES = [
    "/content/drive/MyDrive/PersonaPath/checkpoints/cnn_emotion_phase4.keras",
    PROJECT_ROOT / "models" / "cnn_emotion_phase4.keras",
    PROJECT_ROOT / "cnn_emotion_phase4.keras",
]

VIDEO_CANDIDATES = [
    "/content/Test_emotion_vid1.mp4",
    "/content/drive/MyDrive/PersonaPath/checkpoints/Test_emotion_vid1.mp4",
    PROJECT_ROOT / "Test_emotion_vid1.mp4",
    PROJECT_ROOT / "data" / "Test_emotion_vid1.mp4",
]

def resolve_path(manual_path, candidates, label, override_name):
    if manual_path:
        path = Path(manual_path).expanduser()
        if path.exists():
            return path
        raise FileNotFoundError(f"Manual {label} path does not exist: {path}")

    checked = []
    for candidate in candidates:
        path = Path(candidate).expanduser()
        checked.append(str(path))
        if path.exists():
            return path

    raise FileNotFoundError(
        f"Could not find {label}. Checked:\n  " + "\n  ".join(checked) +
        f"\nSet {override_name} manually near the top of this cell."
    )

def build_emotion_model():
    data_augmentation = tf.keras.Sequential([
        tf.keras.layers.Resizing(224, 224),
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.1),
        tf.keras.layers.RandomZoom(0.1),
    ])

    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False

    model = tf.keras.Sequential([
        tf.keras.layers.InputLayer(input_shape=(None, None, 3)),
        data_augmentation,
        tf.keras.layers.Lambda(lambda x: tf.keras.applications.mobilenet_v2.preprocess_input(x)),
        base_model,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(7, activation='softmax')
    ])
    return model

def validate_model_for_prediction(candidate_model):
    smoke_input = np.zeros((1, 224, 224, 3), dtype=np.float32)
    smoke_preds = candidate_model.predict(smoke_input, verbose=0)
    if smoke_preds.shape[-1] != len(emotion_labels):
        raise ValueError(
            f"Model output shape {smoke_preds.shape} does not match "
            f"{len(emotion_labels)} emotion labels."
        )
    return smoke_preds

def load_emotion_model(model_path):
    print(f"Loading emotion model from: {model_path}")
    load_error = None
    try:
        loaded_model = tf.keras.models.load_model(
            model_path,
            safe_mode=False,
            custom_objects={"tf": tf}
        )
        print("Model loaded with tf.keras.models.load_model().")
        validate_model_for_prediction(loaded_model)
        print("Prediction smoke test passed.")
        return loaded_model
    except Exception as error:
        load_error = error
        print("Direct model load/prediction failed; rebuilding architecture and loading weights instead.")
        print(f"Direct path error: {type(error).__name__}: {error}")

    rebuilt_model = build_emotion_model()
    try:
        rebuilt_model.load_weights(model_path)
        validate_model_for_prediction(rebuilt_model)
    except Exception as rebuild_error:
        raise RuntimeError(
            "Could not load a usable emotion model from the checkpoint. "
            f"Direct error was {type(load_error).__name__}: {load_error}. "
            f"Rebuild/load_weights error was {type(rebuild_error).__name__}: {rebuild_error}"
        ) from rebuild_error

    print("Weights loaded into rebuilt MobileNetV2 emotion model.")
    print("Prediction smoke test passed.")
    return rebuilt_model


MODEL_PATH = resolve_path(MODEL_PATH, MODEL_CANDIDATES, "model", "MODEL_PATH")
INPUT_VIDEO = resolve_path(INPUT_VIDEO, VIDEO_CANDIDATES, "video", "INPUT_VIDEO")
OUTPUT_VIDEO = str(Path(OUTPUT_VIDEO))

model = load_emotion_model(MODEL_PATH)

cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
face_cascade = cv2.CascadeClassifier(cascade_path)
if face_cascade.empty():
    raise RuntimeError(f"Could not load Haar cascade from: {cascade_path}")

print(f"Input video: {INPUT_VIDEO}")
print(f"Output video: {OUTPUT_VIDEO}")

Loading emotion model from: /content/drive/MyDrive/PersonaPath/checkpoints/cnn_emotion_phase4.keras
Model loaded with tf.keras.models.load_model().
Direct model load/prediction failed; rebuilding architecture and loading weights instead.
Direct path error: NameError: Exception encountered when calling Lambda.call().

name 'tf' is not defined

Arguments received by Lambda.call():
  • inputs=tf.Tensor(shape=(1, 224, 224, 3), dtype=float32)
  • mask=None
  • training=False
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Weights loaded into rebuilt MobileNetV2 emotion model.
Prediction smoke test passed.
Input video: /content/drive/MyDrive/PersonaPath/checkpoints/Test_emotion_vid1.mp4
Output video: output_emotion.mp4


In [6]:
def mux_audio_or_finalize(original_video, annotated_silent_video, final_output, preserve_audio=True):
    original_video = str(original_video)
    annotated_silent_video = str(annotated_silent_video)
    final_output = str(final_output)

    ffmpeg = shutil.which("ffmpeg")
    if not preserve_audio or ffmpeg is None:
        if ffmpeg is None and preserve_audio:
            print("ffmpeg not found; keeping muted annotated video.")
        os.replace(annotated_silent_video, final_output)
        return False

    temp_output = str(Path(final_output).with_name(Path(final_output).stem + "_h264_audio.mp4"))
    command = [
        ffmpeg, "-y",
        "-i", annotated_silent_video,
        "-i", original_video,
        "-map", "0:v:0",
        "-map", "1:a:0?",
        "-c:v", "libx264",
        "-pix_fmt", "yuv420p",
        "-c:a", "aac",
        "-shortest",
        temp_output,
    ]

    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode == 0 and Path(temp_output).exists():
        os.replace(temp_output, final_output)
        Path(annotated_silent_video).unlink(missing_ok=True)
        print("Audio preserved when available; output encoded as browser-friendly H.264.")
        return True

    print("Audio muxing failed; keeping muted annotated video.")
    if result.stderr:
        print(result.stderr[-1000:])
    Path(temp_output).unlink(missing_ok=True)
    os.replace(annotated_silent_video, final_output)
    return False


def draw_label(frame, text, x, y, color):
    font = cv2.FONT_HERSHEY_SIMPLEX
    scale = 0.9
    thickness = 2
    (text_w, text_h), baseline = cv2.getTextSize(text, font, scale, thickness)
    top = max(0, y - text_h - baseline - 12)
    cv2.rectangle(frame, (x, top), (x + text_w + 10, top + text_h + baseline + 10), color, -1)
    cv2.putText(frame, text, (x + 5, top + text_h + 3), font, scale, (0, 0, 0), thickness, cv2.LINE_AA)


def process_video(input_path, output_path, preserve_audio=True):
    input_path = Path(input_path)
    output_path = Path(output_path)
    if not input_path.exists():
        raise FileNotFoundError(f"Input video does not exist: {input_path}")

    cap = cv2.VideoCapture(str(input_path))
    if not cap.isOpened():
        raise RuntimeError(f"Error opening video file: {input_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    if width <= 0 or height <= 0:
        cap.release()
        raise RuntimeError(f"Could not read video dimensions from: {input_path}")

    temp_silent = output_path.with_name(output_path.stem + "_silent.mp4")
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(str(temp_silent), fourcc, fps, (width, height))
    if not out.isOpened():
        cap.release()
        raise RuntimeError(f"Could not create output video writer: {temp_silent}")

    stats = {
        "frames": 0,
        "frames_with_faces": 0,
        "detected_faces": 0,
        "predictions": 0,
        "prediction_errors": 0,
    }
    sample_errors = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(
            gray,
            scaleFactor=1.2,
            minNeighbors=5,
            minSize=(60, 60),
            flags=cv2.CASCADE_SCALE_IMAGE,
        )

        if len(faces) > 0:
            stats["frames_with_faces"] += 1
            stats["detected_faces"] += len(faces)

        for (x, y, w, h) in faces:
            x, y, w, h = map(int, (x, y, w, h))
            face_img = frame[y:y+h, x:x+w]
            color = (0, 255, 0)

            try:
                if face_img.size == 0:
                    raise ValueError("Empty face crop")

                face_rgb = cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB)
                face_resized = cv2.resize(face_rgb, (224, 224))
                input_tensor = np.expand_dims(face_resized.astype(np.float32), axis=0)

                preds = model.predict(input_tensor, verbose=0)[0]
                emotion_idx = int(np.argmax(preds))
                emotion = emotion_labels[emotion_idx]
                confidence = float(preds[emotion_idx])

                cv2.rectangle(frame, (x, y), (x+w, y+h), color, 3)
                draw_label(frame, f"{emotion} ({confidence:.2f})", x, y, color)
                stats["predictions"] += 1
            except Exception as error:
                stats["prediction_errors"] += 1
                if len(sample_errors) < 5:
                    sample_errors.append(f"Frame {stats['frames']}, face {(x, y, w, h)}: {type(error).__name__}: {error}")
                cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 0, 255), 3)
                draw_label(frame, "prediction error", x, y, (0, 0, 255))

        out.write(frame)
        stats["frames"] += 1
        if stats["frames"] % 30 == 0:
            print(
                f"Processed {stats['frames']} frames | "
                f"faces: {stats['detected_faces']} | "
                f"predictions: {stats['predictions']} | "
                f"errors: {stats['prediction_errors']}"
            )

    cap.release()
    out.release()

    print("\nProcessing summary:")
    for key, value in stats.items():
        print(f"  {key}: {value}")

    if sample_errors:
        print("\nSample prediction errors:")
        for error in sample_errors:
            print(f"  - {error}")

    if stats["frames"] == 0:
        raise RuntimeError("No frames were read from the input video.")
    if stats["detected_faces"] == 0:
        raise RuntimeError("No faces were detected. Try a clearer/frontal video or tune face detector settings.")
    if stats["predictions"] == 0:
        raise RuntimeError("Faces were detected, but every emotion prediction failed. See errors above.")

    audio_preserved = mux_audio_or_finalize(input_path, temp_silent, output_path, preserve_audio=preserve_audio)
    print(f"Processing complete: {output_path}")
    return {**stats, "audio_preserved": audio_preserved, "output_path": str(output_path)}


result = process_video(INPUT_VIDEO, OUTPUT_VIDEO, preserve_audio=PRESERVE_AUDIO)
result

Processed 30 frames | faces: 31 | predictions: 31 | errors: 0
Processed 60 frames | faces: 61 | predictions: 61 | errors: 0
Processed 90 frames | faces: 91 | predictions: 91 | errors: 0
Processed 120 frames | faces: 121 | predictions: 121 | errors: 0
Processed 150 frames | faces: 151 | predictions: 151 | errors: 0
Processed 180 frames | faces: 181 | predictions: 181 | errors: 0
Processed 210 frames | faces: 211 | predictions: 211 | errors: 0
Processed 240 frames | faces: 241 | predictions: 241 | errors: 0
Processed 270 frames | faces: 271 | predictions: 271 | errors: 0
Processed 300 frames | faces: 301 | predictions: 301 | errors: 0
Processed 330 frames | faces: 331 | predictions: 331 | errors: 0

Processing summary:
  frames: 338
  frames_with_faces: 338
  detected_faces: 339
  predictions: 339
  prediction_errors: 0
Audio preserved when available; output encoded as browser-friendly H.264.
Processing complete: output_emotion.mp4


{'frames': 338,
 'frames_with_faces': 338,
 'detected_faces': 339,
 'predictions': 339,
 'prediction_errors': 0,
 'audio_preserved': True,
 'output_path': 'output_emotion.mp4'}

In [7]:
# Display the output video.
# The processing cell already created an H.264/browser-friendly output when ffmpeg is available.
if os.path.exists(OUTPUT_VIDEO):
    mp4 = open(OUTPUT_VIDEO, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f"""
    <video width="640" controls>
          <source src="{data_url}" type="video/mp4">
    </video>
    """))
else:
    raise FileNotFoundError(f"Output video was not created: {OUTPUT_VIDEO}")